
## Quantum Tree Game



# Cell 1: Import the libraries we need and the backend, then setting the position on the to be defined board



In [1]:
from dataclasses import dataclass, asdict 
from typing import Dict, List, Optional, Set, Tuple, Any
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import matplotlib.pyplot as plt
Position = Tuple[int, int]

### Real Backend 

In [ ]:
class QuantumEngine:
    """
    Handles all underlying Qiskit-related logic, dynamically generating circuits on demand.
    """
    def __init__(self):
        self.simulator = AerSimulator()
        
    def build_and_measure(self, operations):
        """
        Dynamically generate the circuit based on the operation history and measure the final step.
        operations example: [('H', 0), ('CX', 0, 1), ('H', 2)]
        """
        num_qubits = max([op[-1] for op in operations]) + 1 if operations else 1
        qr = QuantumRegister(num_qubits, 'q')
        cr = ClassicalRegister(1, 'c') # Only measure the current latest bit.
        qc = QuantumCircuit(qr, cr)
        
        # Dynamically Applying Quantum Gates
        for op in operations:
            gate = op[0]
            if gate == 'X':
                qc.x(op[1])
            elif gate == 'H':
                qc.h(op[1])
            elif gate == 'CX':
                qc.cx(op[1], op[2]) # op[1] is control, op[2] is target
                
        # Measure the current layer
        qc.measure(num_qubits - 1, 0)
        
        # Execute Simulation
        result = self.simulator.run(qc, shots=1).result()
        counts = result.get_counts()
        
        # Returns the measurement result ('0' or '1'), along with the generated circuit diagram for UI display.
        measured_state = list(counts.keys())[0]
        return measured_state, qc
    
class QuantumEngine:
    def __init__(self, backend_type='simulator', api_key='Mn0JlDfRTzNKBRaQ70-RRaWyE1CodBp-phq7ZkpgO9df', backend_name="ibm_kingston"):
        """
        backend_type: 'simulator' or 'real'
        backend_name: default 'ibm_kingston'
        """
        self.backend_type = backend_type
        
        if self.backend_type == 'real':
            if not api_key:
                raise ValueError("Please provide the api_key！")
            print("Connecting to IBM Quantum Service, please wait...")
            # Set up the IBM Quantum Runtime Service
            self.service = QiskitRuntimeService(channel="ibm_quantum", token=api_key)
            self.backend = self.service.backend(name=backend_name)
            self.sampler = SamplerV2(self.backend)
            print(f"Successfully connected to real quantum computer: {self.backend.name}")
            print(f"Current queue length: {self.backend.status().pending_jobs}")
        else:
            self.simulator = AerSimulator()
            print("Successfully initialized local quantum simulator (AerSimulator).")

    def build_and_measure(self, operations):
        # setup quantum circuit
        num_qubits = max([op[-1] for op in operations]) + 1 if operations else 1
        qr = QuantumRegister(num_qubits, 'q')
        cr = ClassicalRegister(1, 'c')  
        qc = QuantumCircuit(qr, cr)
        
        # Apply quantum gates based on the operations list
        for op in operations:
            gate = op[0]
            if gate == 'X':
                qc.x(op[1])
            elif gate == 'H':
                qc.h(op[1])
            elif gate == 'CX':
                qc.cx(op[1], op[2])
                
        # Measure the last qubit to decide the player's move
        qc.measure(num_qubits - 1, 0)
        
        # ---------------- Core Difference Branching ----------------
        if self.backend_type == 'real':
            print(f"Changing Transpile to {self.backend.name} ...")
            qc_t = transpile(qc, self.backend)
            
            # Set Primitive Unified Bloc (PUB)
            pub = (qc_t,) 
            
            print("Submitting task to IBM Quantum, which may take some time due to hardware queue...")
            job = self.sampler.run([pub], shots=1) 
            print(f"Task submitted successfully! Job ID: {job.job_id()}")
         
            result = job.result()
            
            data = result[0].data
            bits = data.c
            bitstrings = bits.get_bitstrings()
            
            measured_state = bitstrings[0] 
            return measured_state, qc
            
        else:
            # simulator logic remains unchanged
            result = self.simulator.run(qc, shots=1).result()
            counts = result.get_counts()
            measured_state = list(counts.keys())[0]
            return measured_state, qc


## Cell 2: Creating an Object that will save the player state information that will be useful

This cell creates `PlayerState`, which will store:

- the players row and columns currently
- if the player is alive
- if they have won
- their last measurment and direction
- the last wieght they chose

for future use.


In [4]:
@dataclass 
class PlayerState:
    #We need to store all of this information about the player to be grabbed later
    playerId: int
    position: Position
    alive: bool = True
    hasWon: bool = False
    lastMeasurment: Optional[str] = None
    lastDirection: Optional[str] = None
    lastWeight: Optional[int] = None


## Cell 3: Creates the Board the tree will be on

This tree will move along a narrowing path

Inside the board, We will:
- create a constructor with 4 perameters: self (the instance method), height, walls, and goal node to initialize the board
- assign these parameters to be used outside of the method
- create a function that defines where a valid position is inside of the tree
- define where a positon is as a wall or a goal (we save the positions that satisfy this and then check it against the next function) ->
 - -> Check each condition of inside, outside, wall or goal, and return a string for the UI to reference
- Retturns every valid point on the board

In [5]:

class TreeBoard:
    #represents the game board and contains logic for movement, walls, and goals.
    def __init__(
            #constructor that sets up the board with optional walls and goals
        self,
        height: int = 7,
        walls: Optional[Set[Position]] = None, 
        goalNodes: Optional[Set[Position]] = None,

    ): 
        #We can't call the constructor parameters unless we have set data for them
        
        self.height = height #reference height of the board
        self.startRow = height -1 #starting row for players
        self.walls: Set[Position] = walls if walls is not None else set() # This sets the walls on the board
        self.goalNodes: Set[Position] = goalNodes if goalNodes is not None else {(0, 0)} #sets the goal nodes on the board

    #checks if a position is within the bounds of the tree.
    def inTree(self, row: int, col: int) -> bool:

        if row < 0 or row >= self.height:
            return False
        allowed = row
        return -allowed <= col <= allowed
    
     #We must also define what a wall is to be used later
    def wall(self, row: int, col: int) -> bool:
        return (row, col) in self.walls
    
    #Can't forget about defining what the goal is
    def goalNodes(self, row: int, col: int) -> bool: 
        return (row, col) in self.goalNodes
    
    #this will check a position as "wall", "goal", "valid", or "outside_tree".
    def posCheck(self, row: int, col: int) -> str:
        if not self.inTree(row, col):
            return "outside_tree"
        if self.wall(row, col):
            return "wall"
        if (row, col) in self.goalNodes:
            return "goal"
        return "valid"
    
    #this will tell us every valid position on the board we create
    def validNodes(self) -> List[Position]:
        nodes: List[Position] = [] #makes an empty list for each valid position to be added
        for row in range(self.height):
            for col in range(-row, row + 1):
                if self.posCheck(row, col) == "valid": #check if they are valid.
                    nodes.append((row, col)) #add them and return the lists of nodes.
        return nodes


## Cell 4: Runs the actual game logic that will be the backbone of the project

This will Initialise the state of the game and the logic that follows it:

- We start by making another constructor that takes more parameters this time (with some being repeated to pass that data into the board we just built earlier)
- then we check the # of players
- then we assign those values so that they can be used elsewhere besides the method
- in the middle of that we also set the quantum engine, which will be used later when we measure and apply the steps in the movement section
- set the starting position
- save playerstates and so they can be updated later





In [6]:
class TreeGameLogic:
    def __init__(
        self,
        numPlayers: int = 1,
        height: int = 7,
        walls: Optional[Set[Position]] = None,
        goalNodes: Optional[Set[Position]] = None,
        startPositions: Optional[List[Position]] = None,
        backend_type: str = 'simulator',   # new parameter to choose backend
        api_key: Optional[str] = None      # new parameter for API key if using real backend
    ): 
        if numPlayers != 1:
            raise ValueError("numPlayers must be 1.")
        
        self.board = TreeBoard(height=height, walls=walls, goalNodes=goalNodes)
        
        # ------------- real backend initialization -------------
        self.engine = QuantumEngine(backend_type=backend_type, api_key=api_key) 
        # ------------------------------------------------
        
        self.turnNumber = 0
        self.gameOver = False
        self.winnerIds: List[int] = []
        self.numPlayers = numPlayers

        defaultStarts = [(self.board.startRow, 0), (self.board.startRow, 1)] 
        starts = startPositions if startPositions is not None else defaultStarts[:numPlayers]

        self.player: Dict[int, PlayerState] = {} 

        for i in range(numPlayers):
            row, col = starts[i]
            self.player[i + 1] = PlayerState(
                playerId=i + 1,
                position = (row, col)
            )
        self.history = []


## Cell 5: Functions that return clean data for the UI to see

What these methods do:
- `game_state()` returns the overall game state
- `snapshot()` returns every valid board node so a UI can draw the tree
- `preview()` shows the two possible landing spots for one player Based on the node they are at and their weight (since the quantum measurement just decides which way)

Note: The fucntions below will be called through methods through the tree game logic. meaning "TreeGameLogic.apply_measurement_to_position" is the method calling the function "applyMeasurementToPOS"


In [7]:
def gameState(self) -> Dict[str, Any]:
    #returns the current state of the game as a dictionary for display.
    return {
        "turn_number": self.turnNumber,
        "game_over": self.gameOver,
        "winner_ids": self.winnerIds,
        "player": {pid: asdict(state) for pid, state in self.player.items()},
        "walls": list(self.board.walls),
        "goal_nodes": list(self.board.goalNodes),
        "height": self.board.height,
        "num_players": self.numPlayers
    }
TreeGameLogic.gameState = gameState

def snapshot(self) -> List[Dict[str, Any]]:
    #returns a snapshot of the current board state for display (for the UI).
        snapshot: List[Dict[str, Any]] = []
        playerPositions = { 
             #map positions to player ids for quick access when building the snapshot of the board state.
            p.position: p.playerId
            # iterate through player states
            for p in self.player.values()
            #only include players who are alive or have won, since only they matter.
            if p.alive or p.has_won
        }

        for row, col in self.board.validNodes():
            #determine the type of cell (valid, wall, goal) for each position on the board
            cell_type = "valid"
            if (row, col) in self.board.walls:
                cell_type = "wall"
            elif (row, col) in self.board.goalNodes:
                cell_type = "goal"

            snapshot.append({
            #returns all of the previously determined information
                "row": row,
                "col": col,
                "cell_type": cell_type,
                "player_id": playerPositions.get((row, col)),
            })

        return snapshot
TreeGameLogic.snapshot = snapshot

def preview(self, playerId: int, weight: int) -> Dict[str, Any]:
        # Provides a preview of the potential new positions and their classifications for a given player and weight.
        self.validWeight(weight)
        player = self.validPlayer(playerId)
        row, col = player.position
        left_pos = (row - 1, col - weight)
        right_pos = (row - 1, col + weight)

        return {
            #Returns the preview of a single move for one player, including the potential new positions and their location.
            "player_id": playerId,
            "weight": weight,
            "current_position": player.position,
            "left_preview": {
                "position": left_pos,
                "location": self.board.posCheck(*left_pos),
            },
            "right_preview": {
                "position": right_pos,
                "location": self.board.posCheck(*right_pos),
            },
        }
TreeGameLogic.preview = preview


## Cell 6: Functions for checking the weight and making sure the player should be playing the game

What this method does:

- a basic check for making sure the wieght throws no errors
- a check for whether or not a player has already died or has already won

No comments are needed

In [8]:
def validWeight(self, weight: int) -> None:
        if not isinstance(weight, int):
            raise TypeError("weight must be an integer")
        if weight < 1:
            raise ValueError("weight must be at least 1")
TreeGameLogic.validWeight = validWeight

def validPlayer(self, player_id: int) -> PlayerState:
        if player_id not in self.player:
            raise ValueError(f"Invalid player_id: {player_id}")
        player = self.player[player_id]
        if not player.alive:
            raise ValueError(f"Player {player_id} is no longer alive")
        if player.hasWon:
            raise ValueError(f"Player {player_id} has already won")

        return player
TreeGameLogic.validPlayer = validPlayer


## Cell 7: Add movement and rule-checking methods

- `applyMeasurementToPos` converts `0` or `1` into a weighted left/right drift
- `applyNewPos` applies new position and decides if the new position is safe, a wall, outside the tree, or the goal
- `updateGameOver` decides if the whole game should end


- `0` means drift left by the chosen weight
- `1` means drift right by the chosen weight


In [9]:
def applyMeasurementToPos(
        self,
        # Given the current position (row, col), the measured_bit from the quantum measurement, and the chosen weight, calculates the new position and movement direction.
        row: int,
        col: int,
        measured_bit: str,
        weight: int,
    ) -> Tuple[Position, str]:
        
        
        #Moves up one row and drifts left or right by the chosen weight.
        new_row = row - 1

        if measured_bit == "0":
            new_col = col - weight
            direction = f"Up-Left by {weight}"
        elif measured_bit == "1":
            new_col = col + weight
            direction = f"Up-Right by {weight}"
        else:
            raise ValueError(f"Unexpected measured_bit: {measured_bit}")

        return (new_row, new_col), direction
TreeGameLogic.applyMeasurementToPos = applyMeasurementToPos

def applyNewPos(
        # Applies the new position to the player state and determines if they hit a wall, fell outside the tree, reached a goal, or are still alive.
        self,
        player: PlayerState,
        new_pos: Position,
        measured_bit: str,
        direction: str,
        weight: int,
    ) -> Dict[str, Any]:
        
        row, col = new_pos
        status = self.board.posCheck(row, col)
        #Updates the player's position and state.
        player.lastMeasurment = measured_bit
        player.lastDirection = direction
        player.lastWeight = weight

        if status == "outside_tree":
            player.alive = False
            return {
                "result": "fell_outside_tree",
                "position": new_pos,
                "player_alive": False,
                "player_won": False,
            }

        if status == "wall":
            player.alive = False
            player.position = (row, col)
            return {
                "result": "hit_wall",
                "position": new_pos,
                "player_alive": False,
                "player_won": False,
            }

        if status == "goal":
            player.position = (row, col)
            player.hasWon = True
            return {
                "result": "win",
                "position": new_pos,
                "player_alive": True,
                "player_won": True,
            }

        player.position = (row, col)
        return {
            "result": "ok",
            "position": new_pos,
            "player_alive": True,
            "player_won": False,
        }
TreeGameLogic.applyNewPos = applyNewPos

def updateGameOver(self) -> None:
        # Checks if the game is over by looking for any winners or if player is dead.
        alive_players = [p for p in self.player.values() if p.alive]
        winners = [p.playerId for p in self.player.values() if p.hasWon]

        self.winnerIds = winners

        if winners:
            self.gameOver = True
            return {
                 "Congradulations, YOU WON!!"
                 }

        if not alive_players:
            self.gameOver = True
            return {
                 "Game Over"
            }
TreeGameLogic.updateGameOver = updateGameOver


## Cell 8: Add the turn method

- checks whether the game is already over
- reads the token type the user wants
- turns that token into a quantum operation
- asks the backend for a measured state
- converts the measurement into the new position
- updates the player's position
- returns a dictionary that a UI could use


In [10]:
def singleTurn(
        self,
        # apply the chosen token type and weight, and update the game state accordingly.
        player_id: int,
        token_type: str,
        weight: int,
    ) -> Dict[str, Any]:

        if self.gameOver:
            return {"error": "Game is already over."}
        # Validate inputs and get player state.
        self.validWeight(weight)
        player = self.validPlayer(player_id)

        # Prepare the quantum operation history for the backend.
        current_q = self.turnNumber

        # Map token types to quantum gates and build the history.
        if token_type == "Superposition":
            self.history.append(("H", current_q))
            # This means the backend will put the current qubit into a superposition of |0> and |1>, which will lead to a probabilistic move.
        elif token_type == "Entanglement":
            if current_q == 0:
                self.history.append(("H", current_q))
                # If it's the first turn, we can only apply a Hadamard to create a superposition, since there's no previous qubit to entangle with.
            else:
                self.history.append(("CX", current_q - 1, current_q))
                """
                For subsequent turns, we can apply a CNOT gate to entangle the current qubit with the previous one.
                 This means the move will be correlated with the previous move, creating interesting strategic possibilities.
                """
        elif token_type == "Right":
            self.history.append(("X", current_q))
            # Applying an X gate flips the qubit to |1>, which will lead to a move to the right.
        else:
            # "Left" / default leaves the state as |0>, so no gate is added.
            pass

        # The backend will treat the latest qubit (current_q) as the one to measure.
        measured_state, circuit = self.engine.build_and_measure(self.history)

        # Apply the measurement result to determine the new position and direction.
        row, col = player.position
        newPos, direction = self.applyMeasurementToPos(row,col,measured_state,weight,)

        # Apply the new position, update player state, and determine if the game is over.
        condition = self.applyNewPos(player,newPos,measured_state,direction,weight,)

        # Increment turn number and check for game over.
        self.turnNumber += 1
        self.updateGameOver()

        # Return a summary of the turn for the GUI/controller.
        return {
            "mode": "single_player_turn",
            "player_id": player_id,
            "turn_number": self.turnNumber,
            "token_type": token_type,
            "weight": weight,
            "measurement": measured_state,
            "direction": direction,
            "resolution": condition,
            "game_over": self.gameOver,
            "winner_ids": list(self.winnerIds),
            "circuit": circuit,
            "players": {pid: asdict(p) for pid, p in self.player.items()},
        }
TreeGameLogic.singleTurn = singleTurn


## Cell 9: Add a simple text board renderer for debugging

Adds a helper method that prints the tree in plain text.

- draws open spaces as `.`
- draws walls as `X`
- draws the goal as `G`
- draws players as `1` or `2`


In [11]:

def asciiBoard(self) -> str:
    lines = []
    # 
    player_pos_map = {
        p.position: str(p.playerId)
        for p in self.player.values()
        if p.alive or p.hasWon
    }

    max_width = self.board.height * 2 + 1

    for row in range(self.board.height):
        symbols = []
        for col in range(-row, row + 1):
            pos = (row, col)
            if pos in player_pos_map:
                symbol = player_pos_map[pos]
            elif pos in self.board.walls:
                symbol = "X"
            elif pos in self.board.goalNodes:
                symbol = "G"
            else:
                symbol = "."
            symbols.append(symbol)

        row_text = " ".join(symbols)
        lines.append(row_text.center(max_width * 2))

    return "\n".join(lines)

TreeGameLogic.asciiBoard = asciiBoard

game = TreeGameLogic(
    numPlayers=1,
    height=7,
    walls={(4, -2), (3, 1), (2, -1)},
    startPositions=[(6, 0)]
)

print("Initial:")
print(game.asciiBoard())

result = game.singleTurn(
    player_id=1,
    token_type="Superposition",
    weight=1
)

print("\nAfter 1 turn:")
print(game.asciiBoard())


Successfully initialized local quantum simulator (AerSimulator).
Initial:
              G               
            . . .             
          . X . . .           
        . . . . X . .         
      . . X . . . . . .       
    . . . . . . . . . . .     
  . . . . . . 1 . . . . . .   

After 1 turn:
              G               
            . . .             
          . X . . .           
        . . . . X . .         
      . . X . . . . . .       
    . . . . 1 . . . . . .     
  . . . . . . . . . . . . .   



## Cell 10: Added class the UI can call

- Sets up basic information for the UI
- the rest uses the functions made for the UI to reference from above

In [12]:
class GameAPI:
    def __init__(self, backend_type='simulator', api_key=None):
        self.game = TreeGameLogic(  
            numPlayers=1,
            height=7,
            walls={(4, -2), (3, 1), (2, -1)},
            startPositions=[(6, 0)],
            backend_type=backend_type,  # transfer backend choice to the game logic
            api_key=api_key             # transfer API Key to the game logic
        )
        self.gate = "Superposition" 
        self.weight = 1
        self.player_id = 1

    # ... ...
    def set_node_gate(self, gate):
        self.gate = gate
    
    def set_weight(self, weight):
        self.weight = weight

    def preview(self):
        return self.game.preview(playerId=self.player_id, weight=self.weight)

    def singleTurn(self):
        return self.game.singleTurn(player_id=self.player_id, token_type=self.gate, weight=self.weight)

    def snapshot(self):
        return self.game.snapshot()
    
    def gameState(self):
        return self.game.gameState()
    
    def asciiBoard(self):
        return self.game.asciiBoard()

## Cell 11: Calling the GameAPI for a full match

In [17]:
# ==========================================
# Player Configuration
# ==========================================
USE_REAL_BACKEND = True # Set to True to use IBM Quantum real backend, or False to use local simulator
YOUR_IBM_TOKEN = "Mn0JlDfRTzNKBRaQ70-RRaWyE1CodBp-phq7ZkpgO9df"
# ==========================================

if USE_REAL_BACKEND:
    # fix the invalid IBM runtime channel
    def quantum_engine_init(self, backend_type='simulator', api_key=None, backend_name="ibm_kingston"):
        self.backend_type = backend_type
        if self.backend_type == 'real':
            if not api_key:
                raise ValueError("Please provide the api_key!")
            print("Connecting to IBM Quantum Service, please wait...")
            self.service = QiskitRuntimeService(channel="ibm_cloud", token=api_key)
            self.backend = self.service.backend(name=backend_name)
            self.sampler = SamplerV2(self.backend)
            print(f"Successfully connected to real quantum computer: {self.backend.name}")
            print(f"Current queue length: {self.backend.status().pending_jobs}")
        else:
            self.simulator = AerSimulator()
            print("Successfully initialized local quantum simulator (AerSimulator).")

    QuantumEngine.__init__ = quantum_engine_init
else:
    api = GameAPI(backend_type='simulator')

turn = 1
print(api.asciiBoard())

while not api.game.gameOver:   
    print(f"\nTURN {turn}:")

    api.set_node_gate("Superposition")
    api.set_weight(1)

    print("\nPreview:")
    print(api.preview())
    
    result = api.singleTurn()

    print("\nResult:")
    print(result['resolution'])

    print("\nBoard:")
    print(api.asciiBoard())

    turn += 1

print("\nGAME OVER")

              G               
            . . .             
          . X . . .           
        . . . . X . .         
      . . X . . . . . .       
    . . . . . . . . . . .     
  . . . . . . . . . . . . .   

GAME OVER
